In [1]:
!pip install transformers[torch] datasets accelerate evaluate trl peft bitsandbytes rouge_score scipy tqdm -q
!pip install protobuf==4.25.3 -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.3 which is incompatible.


In [ ]:
import torch
import warnings
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)
from peft import PeftModel 
from trl import PPOTrainer, PPOConfig
from trl.models.modeling_value_head import AutoModelForSeq2SeqLMWithValueHead 
import evaluate
import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")

MODEL_NAME = "VietAI/vit5-base"
MAX_SAMPLES = 20000 
MAX_LENGTH = 512

OUTPUT_SFT_DIR = "sft_adapter_summarization" 
OUTPUT_PPO_DIR = "ppo_adapter_summarization"

dataset = load_dataset("nam194/vietnews", split=f"train[:{MAX_SAMPLES}]")

# Chuẩn hóa (Giống hệt SFT)
def preprocess_data(example):
    example["input_text"] = "tóm tắt: " + example["article"]
    example["target_text"] = example["abstract"]
    return example

dataset = dataset.map(
    preprocess_data,
    remove_columns=["guid", "title", "abstract", "article"]
)

# Xáo trộn và Lấy tập train SFT
dataset = dataset.shuffle(seed=42)
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
ppo_run_dataset = split_datasets["train"] # 18000 mẫu

print(f"Đã tải {len(ppo_run_dataset)} mẫu để chạy PPO.")

Đang sử dụng thiết bị: cuda


README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00001-84acb79f6c6547(…):   0%|          | 0.00/170M [00:00<?, ?B/s]

data/validation-00000-of-00001-210cc51bf(…):   0%|          | 0.00/38.3M [00:00<?, ?B/s]

data/test-00000-of-00001-123f98d55067eb7(…):   0%|          | 0.00/38.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/99134 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/22184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22498 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Đã tải 18000 mẫu để chạy PPO.


In [3]:
pip install trl==0.11.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.25.1
    Uninstalling trl-0.25.1:
      Successfully uninstalled trl-0.25.1


In [ ]:
print("Đang giải nén sft_adapter_summary_archive.zip...")
!unzip -o -q sft_adapter_summary_archive.zip

import os
import shutil

if not os.path.exists("sft_adapter_summarization"):
    os.makedirs("sft_adapter_summarization")

files_to_move = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "spiece.model",
    "special_tokens_map.json",
    "training_args.bin", # Nếu có
    "README.md" # Nếu có
]

for file_name in files_to_move:
    source_path = os.path.join(".", file_name)
    destination_path = os.path.join("sft_adapter_summarization", file_name)
    if os.path.exists(source_path) and not os.path.exists(destination_path): 
        shutil.move(source_path, destination_path)

print("Giải nén và sắp xếp tệp hoàn tất. Thư mục 'sft_adapter_summarization' đã sẵn sàng.")

Đang giải nén sft_adapter_summary_archive.zip...
Giải nén và sắp xếp tệp hoàn tất. Thư mục 'sft_adapter_summarization' đã sẵn sàng.


In [ ]:
print("--- Chuẩn bị Huấn luyện PPO ---")

ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=192,
    mini_batch_size=24, 
    gradient_accumulation_steps=8,
    remove_unused_columns=False,
    log_with=None,
    tracker_project_name=None,
    optimize_cuda_cache=True,
    is_encoder_decoder=True,
    seed=42,
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

ppo_peft_model = PeftModel.from_pretrained(base_model, OUTPUT_SFT_DIR, is_trainable=True)

ppo_model = AutoModelForSeq2SeqLMWithValueHead(ppo_peft_model)
ppo_model.is_peft_model = True

ref_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

ref_peft_model = PeftModel.from_pretrained(ref_base_model, OUTPUT_SFT_DIR, is_trainable=False)

ref_model = AutoModelForSeq2SeqLMWithValueHead(ref_peft_model)
ref_model.is_peft_model = True
for param in ref_model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=ppo_run_dataset,
    data_collator=None
)

# Định nghĩa Hàm Phần thưởng (Reward Function)
rouge_metric = evaluate.load("rouge")

def compute_rouge_reward(predictions, references):
    """
    Tính ROUGE-L F-measure làm phần thưởng.
    """
    try:
        rouge_scores = rouge_metric.compute(
            predictions=predictions,
            references=references,
            use_aggregator=False 
        )
        key = 'rougeL' if 'rougeL' in rouge_scores else 'rougeLsum'

        # Scale phần thưởng
        rewards = [score * 10.0 for score in rouge_scores[key]]
        return rewards
    except Exception as e:
        print(f"Lỗi khi tính ROUGE: {e}. Trả về reward 0.")
        return [0.0] * len(predictions)

print("--- Khởi tạo PPO Trainer và Reward Function hoàn tất ---")

--- Chuẩn bị Huấn luyện PPO ---


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


--- Khởi tạo PPO Trainer và Reward Function hoàn tất ---


In [ ]:
print(f"Bắt đầu huấn luyện PPO với {len(ppo_run_dataset)} mẫu...")
print(f"Batch size: {ppo_config.batch_size}, Tích lũy: {ppo_config.gradient_accumulation_steps}")

generation_kwargs = {
    "max_new_tokens": 256,       
    "num_beams": 1,              
    "no_repeat_ngram_size": 2,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

# Sử dụng tqdm để theo dõi tiến độ
pbar = tqdm(ppo_trainer.dataloader, desc=f"PPO Training")

for batch in pbar:
    # batch chứa các cột từ ppo_run_dataset
    prompt_texts = batch['input_text']
    reference_texts = batch['target_text'] # Tóm tắt gốc để tính reward

    # 1. Tokenize prompts
    prompt_tensors = tokenizer(
        prompt_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(ppo_trainer.current_device)

    query_tensors_ids = prompt_tensors['input_ids']

    # 2. Tạo tóm tắt (responses) từ mô hình Policy (ppo_model)
    with torch.no_grad():
        response_tensors = ppo_trainer.generate(
            list(query_tensors_ids), # Chuyển 2D tensor thành danh sách các 1D tensor
            **generation_kwargs
        )

    # 3. Decode tóm tắt (responses)
    response_texts = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

    # 4. Tính phần thưởng (rewards)
    rewards_list = compute_rouge_reward(response_texts, reference_texts)
    rewards_tensors = [torch.tensor(r, device=ppo_trainer.current_device, dtype=torch.float32) for r in rewards_list]

    # 5. Thực hiện PPO step
    response_tensors_cpu = [r.to(torch.device('cpu')) for r in response_tensors]

    try:
        list_query_tensors = [q for q in prompt_tensors['input_ids']]
        list_response_tensors = response_tensors_cpu 

        if not rewards_tensors or len(rewards_tensors) == 0:
            print("Lỗi: rewards_tensors bị rỗng (do tóm tắt rỗng). Bỏ qua bước này.")
            continue

        stats = ppo_trainer.step(list_query_tensors, list_response_tensors, rewards_tensors)

        mean_reward = torch.stack(rewards_tensors).mean().item()
        pbar.set_postfix({"mean_reward": f"{mean_reward:.2f}", "ppo_loss": f"{stats.get('ppo/loss/total', 0):.2f}"})

    except Exception as e:
        print(f"Lỗi trong ppo_trainer.step: {e}")
        torch.cuda.empty_cache()
        continue 

print("--- HUẤN LUYỆN PPO HOÀN TẤT ---")
ppo_model.pretrained_model.save_pretrained(OUTPUT_PPO_DIR)
print(f"Đã lưu mô hình PPO (adapter) vào {OUTPUT_PPO_DIR}")